# What is missing in OBIS — MissingPatterns.jl on live biodiversity records

Two packages, one question. [OBISClient.jl](https://github.com/dantebertuzzi/OBISClient.jl)
pulls occurrence records from [OBIS](https://obis.org), the Ocean Biodiversity Information
System — a programme of the Intergovernmental Oceanographic Commission of UNESCO — and
[MissingPatterns.jl](https://github.com/dantebertuzzi/MissingPatterns.jl) reads the holes in
what comes back.

The holes are the point. An occurrence record is assembled from whatever the provider happened
to record, and a field like depth or life stage is absent far more often than it is present.
Which fields are absent, and whether they are absent *together*, decides what you can actually
ask of the data — usually before you have noticed that the question was decided for you.

> **Neither package is an official OBIS product.** OBISClient.jl is an independent,
> community-maintained client, not affiliated with or endorsed by OBIS, the IOC, or UNESCO. The
> data, the API and the quality control behind them are the work of OBIS and its nodes.

**On Colab, pick the Julia runtime first:** *Runtime ▸ Change runtime type ▸ Julia*.

In [ ]:
using Pkg
Pkg.add(["OBISClient", "MissingPatterns", "DataFrames"])

In [ ]:
using OBISClient, MissingPatterns, DataFrames

## The records

Dolphins — the family Delphinidae — capped at 3000 records. Everything here is live, so the
numbers below will not match the ones committed with this notebook.

In [1]:
recs = OBISClient.occurrence("Delphinidae"; limit = 3000)

OBISTable: 3,000 records, 62 columns (of 580,678 matching)
  endpoint:  occurrence
  accessed:  2026-09-11
  query:     scientificname=Delphinidae
  licenses:  CC-BY-4.0, CC-BY-NC-4.0, CC-BY-SA-4.0, CC0-1.0, unknown
  columns:   id, dataset_id, license, license_url, dataset_citation, occurrenceID, eventID, catalogNumber, … (53 more)

A result carries the whole core schema, most of which is administrative. These are the fields an
analysis would reach for: when, where in the water column, how many animals, under what
conditions, at what precision, and who recorded it.

In [2]:
fields = [:date_start, :depth, :minimumDepthInMeters, :coordinateUncertaintyInMeters,
          :individualCount, :sst, :lifeStage, :sex, :occurrenceStatus,
          :scientificNameID, :institutionCode, :catalogNumber, :datasetName]

df = DataFrame(recs)[:, fields]
size(df)

(3000, 13)

## Column by column

The sparkline is where along the rows the absences sit. It is not a time axis — the records
arrive in the API's own order — but a flat line still says the absence is spread across the
whole pull rather than belonging to one provider's block of records.

In [3]:
missingsummary(df)

 column                    type            missing        %  distribution
 lifeStage                 String             2940   98.00%  ████████████████████
 sex                       String             2917   97.23%  ████████████████████
 minimumDepthInMeters      Float64            1990   66.33%  ▆▆▆▆▆▆▆▆▆▅▆▆▆▆▆▆▆▆▆▆
 depth                     Float64            1985   66.17%  ▆▆▆▆▆▆▆▆▆▅▆▆▆▆▆▆▆▆▆▆
 individualCount           Float64            1881   62.70%  ▆▆▆▆▆▆▆▅▅▆▆▆▅▅▅▆▅▅▅▆
 coordinateUncertaintyIn…  Float64            1603   53.43%  ▅▅▅▅▄▅▄▅▄▅▅▄▅▅▅▄▅▅▅▅
 catalogNumber             String              873   29.10%  ▃▃▃▃▃▃▃▃▃▃▃▃▃▃▂▃▃▃▃▃
 institutionCode           String              458   15.27%  ▁▂▂▂▂▂▂▂▁▂▂▂▂▂▂▂▂▂▂▂
 datasetName               String              409   13.63%  ▁▁▁▁▂▂▂▁▂▂▂▂▁▁▂▂▂▁▁▂
 sst                       Float64              74    2.47%  ▁▁▁▁▁▁▁ ▁▁▁▁▁▁▁▁▁▁▁▁
 date_start                Dates.DateT…         64    2.13%  ▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
 occurrenceStatus       

`lifeStage` and `sex` are absent in the high nineties. That is not a defect: most of these
records are sightings, and you do not sex a dolphin from a boat. It does mean any analysis
conditioning on either is working with a couple of dozen records, whatever the 3000 in the
header suggests.

## Missingness against time

`by` turns the vertical axis into something you can reason about — here the year each record
was collected, taken from the parsed `date_start`. Records with no usable date form the
trailing `∅` group.

In [4]:
missingreport(df; by = :date_start, period = :year, order = :cluster, layout = :classic)

┏━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┓
┃         ┃   98%   ┃   97%   ┃   53%   ┃   29%   ┃   63%   ┃   15%   ┃    2%   ┃   14%   ┃    1%   ┃    0%   ┃   66%   ┃   66%   ┃    2%   ┃
┣━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━┫
┃   row   ┃  life…  ┃   sex   ┃  coor…  ┃  cata…  ┃  indi…  ┃  inst…  ┃  date…  ┃  data…  ┃  occu…  ┃  scie…  ┃  dept…  ┃  mini…  ┃   sst   ┃
┣━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━┫
┃1770-1828┃  █████  ┃  █████  ┃  ▓▓▓▓▓  ┃  ░░░░░  ┃  █████  ┃  ░░░░░  ┃  ░░░░░  ┃  █████  ┃  ░░░░░  ┃  ░░░░░  ┃  █████  ┃  █████  ┃  ░░░░░  ┃
┃1856-1905┃  █████  ┃  █████  ┃  ▓▓▓▓▓  ┃  ▓▓▓▓▓  ┃  █████  ┃  ░░░░░  ┃  ░░░░░  ┃  █████  ┃  ░░░░░  ┃  ░░░░░  ┃  ▓▓▓▓▓  ┃  ▓▓▓▓▓  ┃  ░░░░░  ┃
┃1915-1924┃  █████  ┃  █████  ┃  █████  ┃  ░░░░░  ┃  ▓▓▓▓▓  ┃  ▓▓▓▓▓  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  █████  ┃  █████  ┃  ░░░░░  ┃
┃1927-1950┃  █████  ┃  █████  ┃  █████  ┃  ░░░░░  ┃  ▓▓▓▓▓  ┃  ░░░░░  ┃  ░░░░░  ┃  ▓▓▓▓▓  ┃  ░░░░░  ┃  ░░░░░  ┃  █████  ┃  █████  ┃  ▒▒▒▒▒  ┃
┃1953-1957┃  █████  ┃  █████  ┃  █████  ┃  ░░░░░  ┃  ▓▓▓▓▓  ┃  ░░░░░  ┃  ░░░░░  ┃  ▓▓▓▓▓  ┃  ░░░░░  ┃  ░░░░░  ┃  █████  ┃  █████  ┃  ░░░░░  ┃
┃1964-1965┃  █████  ┃  █████  ┃  █████  ┃  ▓▓▓▓▓  ┃  ▓▓▓▓▓  ┃  ▓▓▓▓▓  ┃  ░░░░░  ┃  ▓▓▓▓▓  ┃  ░░░░░  ┃  ░░░░░  ┃  █████  ┃  █████  ┃  ░░░░░  ┃
┃1967-1968┃  █████  ┃  █████  ┃  ▓▓▓▓▓  ┃  ▓▓▓▓▓  ┃  █████  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ▓▓▓▓▓  ┃  ▓▓▓▓▓  ┃  ░░░░░  ┃
┃1969-1970┃  █████  ┃  █████  ┃  ▓▓▓▓▓  ┃  ▒▒▒▒▒  ┃  ▓▓▓▓▓  ┃  ░░░░░  ┃  ░░░░░  ┃  ▓▓▓▓▓  ┃  ░░░░░  ┃  ░░░░░  ┃  █████  ┃  █████  ┃  ▒▒▒▒▒  ┃
┃1972-1973┃  █████  ┃  ▓▓▓▓▓  ┃  ▓▓▓▓▓  ┃  ░░░░░  ┃  ▓▓▓▓▓  ┃  ░░░░░  ┃  ░░░░░  ┃  ▓▓▓▓▓  ┃  ▓▓▓▓▓  ┃  ░░░░░  ┃  █████  ┃  █████  ┃  ░░░░░  ┃
┃1975-1976┃  █████  ┃  █████  ┃  ▓▓▓▓▓  ┃  ▓▓▓▓▓  ┃  ▓▓▓▓▓  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  █████  ┃  █████  ┃  ░░░░░  ┃
┃1977-1978┃  █████  ┃  █████  ┃  ▓▓▓▓▓  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  █████  ┃  █████  ┃  ░░░░░  ┃
┃1979-1980┃  █████  ┃  █████  ┃  ▒▒▒▒▒  ┃  ░░░░░  ┃  ▒▒▒▒▒  ┃  ░░░░░  ┃  ░░░░░  ┃  ·····  ┃  ·····  ┃  ░░░░░  ┃  █████  ┃  █████  ┃  ░░░░░  ┃
┃1981-1982┃  █████  ┃  █████  ┃  ▓▓▓▓▓  ┃  ▒▒▒▒▒  ┃  ▓▓▓▓▓  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ·····  ┃  ░░░░░  ┃  █████  ┃  █████  ┃  ░░░░░  ┃
┃1983-1984┃  █████  ┃  █████  ┃  █████  ┃  ▒▒▒▒▒  ┃  ▓▓▓▓▓  ┃  ░░░░░  ┃  ░░░░░  ┃  ▒▒▒▒▒  ┃  ░░░░░  ┃  ░░░░░  ┃  █████  ┃  █████  ┃  ░░░░░  ┃
┃1985-1986┃  █████  ┃  █████  ┃  █████  ┃  ▒▒▒▒▒  ┃  ▓▓▓▓▓  ┃  ▒▒▒▒▒  ┃  ░░░░░  ┃  ▒▒▒▒▒  ┃  ·····  ┃  ░░░░░  ┃  █████  ┃  █████  ┃  ░░░░░  ┃
┃1987-1988┃  █████  ┃  █████  ┃  █████  ┃  ▒▒▒▒▒  ┃  ▓▓▓▓▓  ┃  ░░░░░  ┃  ░░░░░  ┃  ▒▒▒▒▒  ┃  ·····  ┃  ░░░░░  ┃  █████  ┃  █████  ┃  ░░░░░  ┃
┃1989-1990┃  █████  ┃  █████  ┃  █████  ┃  ▒▒▒▒▒  ┃  █████  ┃  ·····  ┃  ░░░░░  ┃  ░░░░░  ┃  ·····  ┃  ░░░░░  ┃  █████  ┃  █████  ┃  ░░░░░  ┃
┃1991-1992┃  █████  ┃  █████  ┃  █████  ┃  ▓▓▓▓▓  ┃  █████  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  █████  ┃  █████  ┃  ·····  ┃
┃1993-1994┃  █████  ┃  █████  ┃  █████  ┃  ▓▓▓▓▓  ┃  █████  ┃  ·····  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  █████  ┃  █████  ┃  ░░░░░  ┃
┃1995-1996┃  █████  ┃  █████  ┃  █████  ┃  ▒▒▒▒▒  ┃  ▓▓▓▓▓  ┃  ·····  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  █████  ┃  █████  ┃  ·····  ┃
┃1997-1998┃  █████  ┃  █████  ┃  █████  ┃  ░░░░░  ┃  ▓▓▓▓▓  ┃  ░░░░░  ┃  ░░░░░  ┃  ▒▒▒▒▒  ┃  ░░░░░  ┃  ░░░░░  ┃  █████  ┃  █████  ┃  ·····  ┃
┃1999-2000┃  █████  ┃  █████  ┃  ▓▓▓▓▓  ┃  ░░░░░  ┃  █████  ┃  ·····  ┃  ░░░░░  ┃  ▓▓▓▓▓  ┃  ░░░░░  ┃  ░░░░░  ┃  █████  ┃  █████  ┃  ·····  ┃
┃2001-2002┃  █████  ┃  █████  ┃  █████  ┃  ▒▒▒▒▒  ┃  █████  ┃  ·····  ┃  ░░░░░  ┃  ▒▒▒▒▒  ┃  ░░░░░  ┃  ░░░░░  ┃  █████  ┃  █████  ┃  ·····  ┃
┃2003-2004┃  █████  ┃  █

`missingreport` renders as an HTML grid here, with a tooltip on every cell, and as the Unicode
heatmap in a terminal — same numbers, drawn for whichever medium is asking. In the Unicode
version each glyph is how much of that block is missing: `·` up to 5%, `░` up to 15%, `▒` up to
30%, `▓` up to 50%, `█` above that.

Two readings. `lifeStage` and `sex` are solid black the whole way down, so their absence is not
a matter of old records: nobody is recording them, then or now. And the two depth columns move
as one — same glyph, same period, every row — which the next section quantifies.

What does move is provenance. `catalogNumber`, `institutionCode` and `datasetName` shift period
by period, and so does coordinate uncertainty, because which provider is contributing changes
over time and the fields they fill in change with them.

One caveat on the axis: these are the 3000 records the API happened to return first, not a
sample drawn across years. Read the rows as *what these records look like*, not as a time series
of OBIS as a whole — a distinction the last section puts a number on.

## Which fields go missing together

One row per distinct combination of present and absent fields, most frequent first — R's
`mice::md.pattern()`. Narrowed to seven columns, because the full thirteen make a table too
wide to read.

In [5]:
core = df[:, [:date_start, :depth, :coordinateUncertaintyInMeters,
              :individualCount, :lifeStage, :sex, :catalogNumber]]

missingpatterns(core; max_patterns = 10)

┏━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┓
┃  date…  ┃  dept…  ┃  coor…  ┃  indi…  ┃  life…  ┃   sex   ┃  cata…  ┃    n    ┃    %    ┃  freq   ┃
┣━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━┫
┃  ░░░░░  ┃  █████  ┃  ░░░░░  ┃  ░░░░░  ┃  █████  ┃  █████  ┃  ░░░░░  ┃   615   ┃  20.5%  ┃ ███████ ┃
┃  ░░░░░  ┃  ░░░░░  ┃  █████  ┃  █████  ┃  █████  ┃  █████  ┃  █████  ┃   579   ┃  19.3%  ┃ ███████ ┃
┃  ░░░░░  ┃  █████  ┃  ░░░░░  ┃  █████  ┃  █████  ┃  █████  ┃  ░░░░░  ┃   498   ┃  16.6%  ┃ ██████  ┃
┃  ░░░░░  ┃  █████  ┃  █████  ┃  ░░░░░  ┃  █████  ┃  █████  ┃  ░░░░░  ┃   443   ┃  14.8%  ┃ █████   ┃
┃  ░░░░░  ┃  ░░░░░  ┃  █████  ┃  █████  ┃  █████  ┃  █████  ┃  ░░░░░  ┃   256   ┃  8.5%   ┃ ███     ┃
┃  ░░░░░  ┃  █████  ┃  █████  ┃  █████  ┃  █████  ┃  █████  ┃  ░░░░░  ┃   221   ┃  7.4%   ┃ ███     ┃
┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  █████  ┃  █████  ┃  █████  ┃  █████  ┃   120   ┃ 

The same question as a correlation: ϕ between every pair of missingness masks, positive where
two fields go absent together.

In [6]:
missingcooccurrence(df)

┏━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┓
┃    ϕ    ┃  date…  ┃  dept…  ┃  mini…  ┃  coor…  ┃  indi…  ┃   sst   ┃  life…  ┃   sex   ┃  occu…  ┃  scie…  ┃  inst…  ┃  cata…  ┃  data…  ┃
┣━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━┫
┃  date…  ┃    —    ┃  0.09   ┃  0.09   ┃  0.09   ┃  0.08   ┃  -0.02  ┃  0.02   ┃  -0.05  ┃  0.17   ┃  0.07   ┃  0.21   ┃  0.12   ┃  0.35   ┃
┃  dept…  ┃  0.09   ┃    —    ┃  1.00   ┃  -0.43  ┃  -0.54  ┃  0.10   ┃  0.10   ┃  0.03   ┃  0.03   ┃  0.00   ┃  -0.30  ┃  -0.69  ┃  0.12   ┃
┃  mini…  ┃  0.09   ┃  1.00   ┃    —    ┃  -0.42  ┃  -0.53  ┃  0.10   ┃  0.10   ┃  0.03   ┃  0.03   ┃  0.00   ┃  -0.29  ┃  -0.69  ┃  0.12   ┃
┃  coor…  ┃  0.09   ┃  -0.43  ┃  -0.42  ┃    —    ┃  0.17   ┃  -0.05  ┃  0.08   ┃  0.12   ┃  0.06   ┃  -0.03  ┃  0.20   ┃  0.27   ┃  -0.08  ┃
┃  ind

Two things stand out. `depth` and `minimumDepthInMeters` sit at ϕ = 1.00 — not correlated but
identical, because OBIS's interpreted `depth` is absent in exactly the records where the
provider gave no minimum depth. Carrying both into an analysis adds a column and no information.

`depth` against `catalogNumber` runs strongly *negative*, which is the more interesting
direction: those two are missing in *different* records. The likeliest reading is that the pull
mixes kinds of provider — a museum specimen has a catalogue number and no depth, a survey has
the depth and no catalogue number — so what looks like one table is two populations stacked on
top of each other, and nothing in the table says so.

## What complete-case analysis costs

`missingrows` prices listwise deletion as the table stands. The `0` line is what survives
`dropmissing`.

In [7]:
missingrows(df)

 missing/row  rows        %  distribution
 2               1    0.03%  █
 3              36    1.20%  █
 4             722   24.07%  █████████████
 5            1622   54.07%  ██████████████████████████████
 6             395   13.17%  ███████
 7             148    4.93%  ███
 8              25    0.83%  █
 9               9    0.30%  █
 10             42    1.40%  █
 0 complete rows (0.00%) ┊ 3000 with ≥1 missing (100.00%) ┊ 9 distinct counts across 13 columns


There is usually no `0` line at all: across these thirteen fields, not one record in three
thousand is complete. `dropmissing` would return an empty table, and a model fitted on
"the data" would silently be fitted on whatever subset its formula happened to spare.

`missingdrop` prices the alternative — trading a field for records — and names the field worth
trading first.

In [8]:
missingdrop(df)

 drop                           cols  complete        %  distribution
 —                                13         0    0.00%  
 lifeStage                        12         0    0.00%  
 sex                              11         1    0.03%  
 datasetName                      10         3    0.10%  
 individualCount                   9         6    0.20%  
 catalogNumber                     8       118    3.93%  █
 coordinateUncertaintyInMeters     7       703   23.43%  ███████
 institutionCode                   6      1003   33.43%  ██████████
 minimumDepthInMeters              5      1008   33.60%  ██████████
 depth                             4      2849   94.97%  ████████████████████████████  ◀ most complete-case cells
 sst                               3      2922   97.40%  █████████████████████████████
 date_start                        2      2980   99.33%  ██████████████████████████████
 occurrenceStatus                  1      2997   99.90%  ██████████████████████████████
 0 

## Against OBIS's own numbers

The package is reading 3000 records; OBIS can report the same absences across the entire query
without sending any records at all. `statistics_qc` is the cheap check, and the comparison is
worth making: the first 3000 records are not a random sample of the 580-odd thousand, and the
gap between the two columns is how badly that shows.

In [9]:
qc    = OBISClient.statistics_qc("Delphinidae")
total = OBISClient.estimate_size(scientificname = "Delphinidae")
stats = Dict(string(r.column) => r for r in missingstats(df))

# OBIS reports a missing count only for the fields its QC pipeline checks
shared = [f for f in sort(collect(keys(stats)))
          if haskey(qc["fields"], f) && haskey(qc["fields"][f], "missing")]

DataFrame(
    field      = shared,
    sample_pct = [round(stats[f].pct; digits = 1) for f in shared],
    obis_pct   = [round(100 * qc["fields"][f]["missing"] / total; digits = 1) for f in shared],
)

Row,field,sample_pct,obis_pct
,String,Float64,Float64
1,coordinateUncertaintyInMeters,53.4,43.5
2,minimumDepthInMeters,66.3,73.3
3,occurrenceStatus,0.6,0.8
4,scientificNameID,0.1,0.4


Where the two disagree, believe OBIS: `sample_pct` is 3000 records deep and `obis_pct` is the
whole query. Note that the sample misses in *both* directions — it overstates the absence of one
field and understates another — so it is not a bias you could correct for by eye. Treat the
sample as a shape, not as an estimate — and if you need the real
distribution, pull more records or go to the bulk export, which
[Large queries](https://dantebertuzzi.github.io/OBISClient.jl/stable/large-queries/) covers.

## The licence did not go away

None of the above changes what the data is licensed for. Rights and citations travel on the
result, and the most restrictive licence in a multi-dataset pull governs the whole of it.

In [10]:
OBISClient.licenses(recs)

OBISTable: 5 records, 7 columns
  endpoint:  licenses
  accessed:  2026-09-11
  query:     scientificname=Delphinidae
  licenses:  CC-BY-4.0, CC-BY-NC-4.0, CC-BY-SA-4.0, CC0-1.0, unknown
  columns:   license, datasets, records, permits_redistribution, permits_commercial_use, requires_attribution, url

```julia
write("obis-references.bib", OBISClient.citations(recs; format = :bibtex))
```

Cite the datasets you used, with the date you accessed them.

## Where to go next

- [MissingPatterns.jl](https://dantebertuzzi.github.io/MissingPatterns.jl/stable) — every entry
  point and every keyword, and
  [`getting-started.ipynb`](https://colab.research.google.com/github/dantebertuzzi/MissingPatterns.jl/blob/main/notebooks/getting-started.ipynb)
  for the tour on a table whose missingness was put there on purpose
- [OBISClient.jl](https://dantebertuzzi.github.io/OBISClient.jl/stable) — the filters, the other
  endpoints, and the bulk export
- [Interpreting OBIS data](https://dantebertuzzi.github.io/OBISClient.jl/stable/interpreting/) —
  the other half of this notebook's question: what a default query silently excludes, and why
  record counts measure sampling effort as much as biology